# Visualize Diffleop SDF Outputs

This notebook displays generated Diffleop molecules from SDF files in 3D using `py3Dmol`. It is intended to be run from the `Diffleop` directory or from this notebook with the path setup cell below.


In [1]:
from pathlib import Path
import sys

def find_diffleop_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "configs").is_dir() and (path / "scripts").is_dir() and (path / "data").is_dir():
            return path
    raise RuntimeError("Could not find the Diffleop root directory")

ROOT = find_diffleop_root()
sys.path.insert(0, str(ROOT))
print(ROOT)


/Volumes/SATECHI_DISK_Media/Projects/huggingface/gpcr-diffleop/Diffleop


In [2]:
try:
    import py3Dmol
except ImportError as exc:
    raise ImportError("Install py3Dmol first: mamba run -n diffleop pip install py3Dmol") from exc

from rdkit import Chem
print("py3Dmol and RDKit are available")


py3Dmol and RDKit are available


## Select an Output

Change `OUTPUT_ID` to inspect another generated molecule under `outputs/sampling_dec_smoke_001/sdf/`.


In [3]:
RUN_DIR = ROOT / "outputs" / "sampling_dec_smoke_001"
OUTPUT_ID = "0"
SAMPLE_INDEX = 0

sample_dir = RUN_DIR / "sdf" / OUTPUT_ID
sdf_path = sample_dir / f"{SAMPLE_INDEX}.sdf"
retain_path = sample_dir / "smiles_retain.smi"

print("sample_dir:", sample_dir)
print("sdf_path:", sdf_path, sdf_path.exists())
print("retain_path:", retain_path, retain_path.exists())


sample_dir: /Volumes/SATECHI_DISK_Media/Projects/huggingface/gpcr-diffleop/Diffleop/outputs/sampling_dec_smoke_001/sdf/0
sdf_path: /Volumes/SATECHI_DISK_Media/Projects/huggingface/gpcr-diffleop/Diffleop/outputs/sampling_dec_smoke_001/sdf/0/0.sdf True
retain_path: /Volumes/SATECHI_DISK_Media/Projects/huggingface/gpcr-diffleop/Diffleop/outputs/sampling_dec_smoke_001/sdf/0/smiles_retain.smi True


## Input Metadata

`smiles_retain.smi` records the retained scaffold, masked fragment, original ligand, source ligand SDF, and protein pocket PDB for each demo input.


In [4]:
def read_retain_metadata(path):
    lines = path.read_text().splitlines()
    keys = ["retain_smi", "mask_smi", "original_ligand_smi", "ligand_file", "protein_pocket_file"]
    return dict(zip(keys, lines))

metadata = read_retain_metadata(retain_path)
metadata


{'retain_smi': 'Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(C[*:1])cc32)c1',
 'mask_smi': 'c1ccc([*:1])cc1',
 'original_ligand_smi': 'Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(Cc4ccccc4)cc32)c1',
 'ligand_file': 'XIAP_HUMAN_242_355_0/4hy0_A_rec_5m6e_7ht_lig_tt_docked_1.sdf',
 'protein_pocket_file': 'XIAP_HUMAN_242_355_0/4hy0_A_rec_5m6e_7ht_lig_tt_docked_1_pocket10.pdb'}

## Generated Molecule Only


In [5]:
sdf = sdf_path.read_text()

view = py3Dmol.view(width=800, height=600)
view.addModel(sdf, "sdf")
view.setStyle({"stick": {"radius": 0.18, "colorscheme": "greenCarbon"}})
view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Generated Molecule with Protein Pocket

This cell displays the protein pocket only if the raw PDB file exists locally. The checked-in demo data may include only processed LMDB/PT files, so the cell falls back to ligand-only display when the PDB is missing.


In [6]:
protein_rel = Path(metadata["protein_pocket_file"])
protein_candidates = [
    ROOT / "data" / "demo" / "dec" / protein_rel,
    ROOT / protein_rel,
    sample_dir / protein_rel.name,
]
protein_path = next((p for p in protein_candidates if p.exists()), None)

if protein_path is None:
    print("Protein pocket PDB was not found locally.")
    print("The demo dataset in this repository contains processed LMDB/PT files, but not the raw PDB path recorded in smiles_retain.smi:")
    print(metadata["protein_pocket_file"])
    print("Showing the generated ligand only.")

    view = py3Dmol.view(width=900, height=650)
    view.addModel(sdf, "sdf")
    view.setStyle({"stick": {"radius": 0.18, "colorscheme": "greenCarbon"}})
    view.zoomTo()
    view.show()
else:
    print("protein_path:", protein_path)
    pdb = protein_path.read_text()

    view = py3Dmol.view(width=900, height=650)
    view.addModel(pdb, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.25, "color": "white"}, {"model": 0})

    view.addModel(sdf, "sdf")
    view.setStyle({"model": 1}, {"stick": {"radius": 0.18, "colorscheme": "greenCarbon"}})

    view.zoomTo({"model": 1})
    view.show()


Protein pocket PDB was not found locally.
The demo dataset in this repository contains processed LMDB/PT files, but not the raw PDB path recorded in smiles_retain.smi:
XIAP_HUMAN_242_355_0/4hy0_A_rec_5m6e_7ht_lig_tt_docked_1_pocket10.pdb
Showing the generated ligand only.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## List Available Generated SDF Files


In [7]:
for path in sorted((RUN_DIR / "sdf").glob("*/*.sdf")):
    print(path.relative_to(ROOT))


outputs/sampling_dec_smoke_001/sdf/0/0.sdf
outputs/sampling_dec_smoke_001/sdf/1/0.sdf
